# Dendritron Transformer v0.4.2: Registry Repair

v0.4.1 registered four independent functional memories. Only the
character-level `first_letter_order` pack was quarantined.

v0.4.2 preserves those four packs as software assets, replaces only the failed
memory with `endpoint_match`, and proceeds into the complete routing, binding,
reload, deletion, and reinstall gates.


## Required clean restart

Before running this corrected version, choose:

**Runtime → Restart session**

The previous run imported PEFT while Colab's incompatible
`torchao==0.10.0` was installed. This notebook removes that optional package
before PEFT is imported. A fresh process ensures neither `torchao` nor PEFT is
cached from the failed run.


## What v0.4.1 changes

- 320 training, 100 validation, and 120 test prompts per memory.
- Up to eight epochs with a three-epoch minimum.
- Cosine learning-rate decay with a 10% floor.
- Registration is an integer test: at least 80 of 100 validation examples.
- Every pack trains even when an earlier pack misses registration.
- Failed packs are quarantined and bundled with their complete learning curves.

The frozen backbone, LoRA-only gradients, autonomous address proposal, PPCA
binding, full backbone SHA-256, reload logits, deletion, reinstall, and adapter
hash tests remain unchanged.


In [ ]:
# Standard LoRA does not use torchao. Colab currently ships an old optional
# torchao build that newer PEFT releases reject during adapter injection.
%pip uninstall -y torchao

# Keep Colab's own dependency contract intact.
%pip install -q --upgrade-strategy only-if-needed \
    "transformers>=4.55,<6" \
    "peft>=0.19.1,<1" \
    "accelerate>=1.5,<2" \
    "safetensors>=0.5" \
    "scikit-learn>=1.5,<1.8" \
    "joblib>=1.4" \
    "pandas==2.2.2" \
    "matplotlib>=3.8,<3.11"


In [ ]:
# Preflight: fail here with a clear message before any training begins.
import sys
import torch
import transformers
import peft
import accelerate
import sklearn
import pandas as pd

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("scikit-learn:", sklearn.__version__)

import importlib.util
import importlib.metadata

torchao_spec = importlib.util.find_spec("torchao")
if torchao_spec is not None:
    torchao_version = importlib.metadata.version("torchao")
    raise RuntimeError(
        "torchao is still installed after the uninstall cell "
        f"(version {torchao_version}). Choose Runtime > Restart session, "
        "then rerun from the first cell."
    )
print("Optional torchao check: PASS (not installed; standard LoRA does not need it)")

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU is attached. In Colab choose Runtime > Change runtime type > GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "GB",
)
print("BF16 supported:", torch.cuda.is_bf16_supported())

# Detect a poisoned CUDA context before downloading or training the model.
try:
    _cuda_probe = torch.tensor([1.0, 2.0], device="cuda")
    _cuda_probe_result = (_cuda_probe * 2).sum()
    torch.cuda.synchronize()
    assert float(_cuda_probe_result.cpu().item()) == 6.0
    print("CUDA context health check: PASS")
except Exception as error:
    raise RuntimeError(
        "This Colab GPU process is still poisoned by a previous device-side "
        "assert. Choose Runtime > Restart session and run again."
    ) from error


## Dependency and adapter-injection smoke test

Before the 360M model is downloaded, this cell wraps a tiny local linear model
with PEFT LoRA. It verifies that the installed PEFT stack can inject an adapter
without touching `torchao`.

This specifically catches the environment failure from the previous run before
any Dendritron training begins.


In [ ]:
# Verify PEFT LoRA injection independently of SmolLM2.
import torch
import torch.nn as nn
from peft import LoraConfig, TaskType, get_peft_model

class TinyCausalLikeModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.config = type(
            "Config",
            (),
            {
                "model_type": "custom",
                "tie_word_embeddings": False,
                "get": lambda self, key, default=None: getattr(self, key, default),
            },
        )()
        self.q_proj = nn.Linear(16, 16, bias=False)

    def forward(self, input_ids=None, **kwargs):
        hidden = torch.zeros(2, 16)
        return self.q_proj(hidden)

tiny_model = TinyCausalLikeModel()
tiny_lora = get_peft_model(
    tiny_model,
    LoraConfig(
        r=2,
        lora_alpha=4,
        target_modules=["q_proj"],
        bias="none",
        task_type=None,
    ),
)
trainable = sum(
    parameter.numel()
    for parameter in tiny_lora.parameters()
    if parameter.requires_grad
)
assert trainable > 0
print("PEFT LoRA injection smoke test: PASS")
print("Tiny trainable adapter parameters:", trainable)
del tiny_lora, tiny_model


In [ ]:
# Configuration and optional registry repair.
from pathlib import Path

RUN_PROFILE = "showcase"
MOUNT_GOOGLE_DRIVE = False

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/dendritron_smollm2_360m_v4_2"
    PRIOR_V4_1_DIR = "/content/drive/MyDrive/dendritron_smollm2_360m_v4_1"
else:
    OUTPUT_DIR = "/content/dendritron_smollm2_360m_v4_2"
    PRIOR_V4_1_DIR = "/content/dendritron_smollm2_360m_v4_1"

QUICK_MODE = RUN_PROFILE == "showcase"
BOOTSTRAP_DIR = (
    PRIOR_V4_1_DIR
    if Path(PRIOR_V4_1_DIR).exists()
    else None
)

print("Profile:", RUN_PROFILE)
print("Output directory:", OUTPUT_DIR)
print("Prior registry:", PRIOR_V4_1_DIR)
print("Bootstrap source:", BOOTSTRAP_DIR)

if BOOTSTRAP_DIR is None:
    print("Prior registry not found; all five packs will train.")


## The repaired five-memory registry

| Memory | Function |
|---|---|
| `sum_threshold` | Sum of five integers is at least 250 |
| `vowel_majority` | Vowels are more than half of a string |
| `balanced_brackets` | Parenthesis sequence is valid and balanced |
| `alternating_sequence` | Sequence alternates between two values |
| `endpoint_match` | First and final color tokens are identical |

`endpoint_match` is relational like the failed task, but its evidence remains
explicit under subword tokenization.


In [ ]:
# Write the v0.4.2 implementation into Colab.
from pathlib import Path

BENCHMARK_SOURCE = '\nfrom __future__ import annotations\n\nimport argparse\nimport contextlib\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport os\nimport random\nimport re\nimport shutil\nimport time\nimport zipfile\nfrom dataclasses import asdict, dataclass, field\nfrom pathlib import Path\nfrom typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn.functional as F\nfrom sklearn.decomposition import PCA\nfrom sklearn.linear_model import LogisticRegression\nfrom torch.utils.data import DataLoader, Dataset\nfrom transformers import AutoModelForCausalLM, AutoTokenizer, set_seed\nfrom peft import LoraConfig, PeftModel, TaskType, get_peft_model\n\n\nMEMORY_IDS = (\n    "sum_threshold",\n    "vowel_majority",\n    "balanced_brackets",\n    "alternating_sequence",\n    "endpoint_match",\n)\n\nOBJECTIVE_VERSION = "restricted_binary_next_token_v2_cosine"\nRUNTIME_VERSION = "dendritron-smollm2-v0.4.2"\n\n\n@dataclass\nclass Config:\n    model_id: str = "HuggingFaceTB/SmolLM2-360M"\n    output_dir: str = "/content/dendritron_smollm2_360m_v4_2"\n    seed: int = 7\n    quick_mode: bool = False\n    bootstrap_from_dir: Optional[str] = None\n\n    train_examples_per_memory: int = 640\n    validation_examples_per_memory: int = 160\n    test_examples_per_memory: int = 240\n\n    max_length: int = 96\n    train_epochs: int = 8\n    minimum_train_epochs: int = 3\n    early_stopping_patience: int = 3\n    registration_min_accuracy: float = 0.80\n    batch_size: int = 16\n    gradient_accumulation: int = 2\n    learning_rate: float = 3e-4\n    weight_decay: float = 0.01\n    warmup_fraction: float = 0.08\n    minimum_learning_rate_fraction: float = 0.10\n    max_grad_norm: float = 1.0\n\n    lora_rank: int = 8\n    lora_alpha: int = 16\n    lora_dropout: float = 0.05\n    lora_targets: Tuple[str, ...] = (\n        "q_proj",\n        "k_proj",\n        "v_proj",\n        "o_proj",\n        "gate_proj",\n        "up_proj",\n        "down_proj",\n    )\n\n    tap_fraction: float = 0.35\n    pca_dim: int = 64\n    verifier_rank: int = 8\n    covariance_floor: float = 1e-5\n    address_scale: float = 3.0\n    temperature_steps: int = 120\n    temperature_learning_rate: float = 0.05\n\n    efficient_singleton_accuracy: float = 0.97\n    reliable_candidate_coverage: float = 0.97\n    critical_nominal_coverage: float = 0.99\n    raps_penalty: float = 0.02\n    raps_regularization_rank: int = 1\n\n    known_acceptance_target: float = 0.98\n    coordinate_batch_size: int = 32\n    inference_batch_size: int = 48\n    reload_logit_tolerance: float = 2e-3\n\n    def finalized(self) -> "Config":\n        if self.quick_mode:\n            self.train_examples_per_memory = 320\n            self.validation_examples_per_memory = 100\n            self.test_examples_per_memory = 120\n            self.train_epochs = 8\n            self.minimum_train_epochs = 3\n            self.early_stopping_patience = 3\n            self.batch_size = 16\n            self.gradient_accumulation = 1\n            self.learning_rate = 4e-4\n            self.coordinate_batch_size = 24\n            self.inference_batch_size = 24\n            self.pca_dim = 32\n            self.verifier_rank = 6\n        return self\n\n\nGATE_THRESHOLDS = {\n    "minimum_pack_validation_accuracy": 0.80,\n    "oracle_accuracy": 0.90,\n    "reliable_accuracy": 0.84,\n    "critical_accuracy": 0.87,\n    "critical_oracle_retention": 0.94,\n    "critical_average_candidates": 3.20,\n    "address_top2_coverage": 0.90,\n    "old_memory_prediction_retention": 1.0,\n    "backbone_hash_retention": 1.0,\n    "checkpoint_prediction_equivalence": 1.0,\n    "checkpoint_candidate_equivalence": 1.0,\n    "checkpoint_max_logit_delta": 2e-3,\n    "uninstall_selection_exclusion": 1.0,\n    "reinstall_prediction_equivalence": 1.0,\n    "reinstall_max_logit_delta": 2e-3,\n    "adapter_hash_equivalence": 1.0,\n}\n\n\ndef seed_everything(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    set_seed(seed)\n\n\ndef atomic_json_dump(payload: Dict[str, Any], path: Path) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    with temporary.open("w", encoding="utf-8") as handle:\n        json.dump(payload, handle, indent=2)\n    temporary.replace(path)\n\n\ndef append_progress(output_dir: Path, stage: str, **kwargs: Any) -> None:\n    path = output_dir / "progress.json"\n    if path.exists():\n        payload = json.loads(path.read_text(encoding="utf-8"))\n    else:\n        payload = {"events": []}\n    payload["events"].append(\n        {\n            "time": time.strftime("%Y-%m-%d %H:%M:%S"),\n            "stage": stage,\n            **kwargs,\n        }\n    )\n    payload["last_stage"] = stage\n    atomic_json_dump(payload, path)\n\n\ndef select_dtype() -> torch.dtype:\n    if not torch.cuda.is_available():\n        raise RuntimeError(\n            "This benchmark requires a Colab GPU. Select Runtime > Change runtime type > GPU."\n        )\n    if torch.cuda.is_bf16_supported():\n        return torch.bfloat16\n    return torch.float16\n\n\ndef validate_torchao_environment() -> Dict[str, Any]:\n    """\n    Standard LoRA does not require torchao.\n\n    Some Colab images include an old torchao build. New PEFT releases inspect\n    that optional package while dispatching LoRA modules and reject versions\n    <=0.16.0. Prefer no torchao at all for this non-quantized benchmark.\n    """\n    try:\n        version_text = importlib.metadata.version("torchao")\n    except importlib.metadata.PackageNotFoundError:\n        return {\n            "torchao_installed": False,\n            "torchao_version": None,\n            "status": "PASS: optional torchao is absent",\n        }\n\n    numeric_parts = []\n    for part in version_text.split("."):\n        match = re.match(r"(\\d+)", part)\n        if match is None:\n            break\n        numeric_parts.append(int(match.group(1)))\n    while len(numeric_parts) < 3:\n        numeric_parts.append(0)\n    version_tuple = tuple(numeric_parts[:3])\n\n    if version_tuple <= (0, 16, 0):\n        raise RuntimeError(\n            f"Incompatible optional torchao version {version_text} is installed. "\n            "This benchmark uses ordinary BF16/FP16 LoRA and does not need "\n            "torchao. Run `pip uninstall -y torchao`, restart the Colab session, "\n            "and run the notebook again."\n        )\n\n    return {\n        "torchao_installed": True,\n        "torchao_version": version_text,\n        "status": "PASS: torchao version is compatible",\n    }\n\n\ndef gpu_summary() -> Dict[str, Any]:\n    properties = torch.cuda.get_device_properties(0)\n    return {\n        "device": torch.cuda.get_device_name(0),\n        "total_memory_gb": round(properties.total_memory / 1024**3, 2),\n        "bf16": bool(torch.cuda.is_bf16_supported()),\n        "torch_version": torch.__version__,\n    }\n\n\ndef random_word(rng: np.random.Generator, minimum: int = 4, maximum: int = 9) -> str:\n    alphabet = np.asarray(list("abcdefghijklmnopqrstuvwxyz"))\n    length = int(rng.integers(minimum, maximum + 1))\n    return "".join(rng.choice(alphabet, size=length).tolist())\n\n\ndef balanced_parentheses(rng: np.random.Generator, pairs: int) -> str:\n    result: List[str] = []\n    opens = 0\n    closes = 0\n    while closes < pairs:\n        if opens < pairs and (opens == closes or rng.random() < 0.58):\n            result.append("(")\n            opens += 1\n        else:\n            result.append(")")\n            closes += 1\n    return "".join(result)\n\n\ndef make_example(memory_index: int, label: int, rng: np.random.Generator) -> str:\n    """Generate five functions inside one common prompt envelope."""\n    rule: str\n    payload: str\n\n    if memory_index == 0:\n        while True:\n            values = rng.integers(0, 100, size=5).astype(int)\n            outcome = int(values.sum() >= 250)\n            if outcome == label:\n                break\n        rule = "The sum of the five integers is at least 250."\n        payload = f"Numbers: {\', \'.join(str(value) for value in values)}"\n\n    elif memory_index == 1:\n        vowels = np.asarray(list("aeiou"))\n        consonants = np.asarray(list("bcdfghjklmnpqrstvwxyz"))\n        length = 12\n        vowel_count = int(rng.integers(7, 11)) if label == 1 else int(rng.integers(1, 6))\n        characters = np.concatenate([\n            rng.choice(vowels, size=vowel_count),\n            rng.choice(consonants, size=length - vowel_count),\n        ])\n        rng.shuffle(characters)\n        text = "".join(characters.tolist())\n        rule = "Vowels are strictly more than half of the lowercase string."\n        payload = f"String: {text}"\n\n    elif memory_index == 2:\n        pairs = int(rng.integers(3, 8))\n        text = balanced_parentheses(rng, pairs)\n        if label == 0:\n            if rng.random() < 0.5:\n                text = ")" + text[1:]\n            else:\n                text = text[:-1] + "("\n        rule = "Every prefix is valid and the complete parenthesis string is balanced."\n        payload = f"Sequence: {text}"\n\n    elif memory_index == 3:\n        first, second = rng.choice(np.arange(1, 20), size=2, replace=False).astype(int)\n        values = np.asarray([\n            first if position % 2 == 0 else second\n            for position in range(10)\n        ], dtype=int)\n        if label == 0:\n            position = int(rng.integers(1, 9))\n            replacement_pool = [\n                value for value in range(1, 20)\n                if value not in (first, second)\n            ]\n            values[position] = int(rng.choice(replacement_pool))\n        rule = "The sequence alternates exactly between two distinct integers."\n        payload = f"Sequence: {\' \'.join(str(value) for value in values)}"\n\n    elif memory_index == 4:\n        # Token-level endpoint relation. The signal remains explicit after\n        # subword tokenization.\n        symbols = np.asarray(\n            [\n                "red",\n                "blue",\n                "green",\n                "yellow",\n                "black",\n                "white",\n                "orange",\n                "purple",\n            ]\n        )\n        first_symbol = str(rng.choice(symbols))\n        middle = [\n            str(value)\n            for value in rng.choice(\n                symbols,\n                size=6,\n                replace=True,\n            )\n        ]\n        if label == 1:\n            final_symbol = first_symbol\n        else:\n            final_symbol = str(\n                rng.choice(\n                    symbols[\n                        symbols != first_symbol\n                    ]\n                )\n            )\n        sequence = [\n            first_symbol,\n            *middle,\n            final_symbol,\n        ]\n        rule = (\n            "The first and final color tokens "\n            "are identical."\n        )\n        payload = (\n            "Color sequence: "\n            + " | ".join(sequence)\n        )\n\n    else:\n        raise ValueError(memory_index)\n\n    return (\n        "Evaluate one rule against one input. Return yes when the rule is true "\n        "and no when it is false.\\n"\n        f"Rule: {rule}\\n"\n        f"Input:\\n{payload}\\n"\n        "Answer:"\n    )\n\n\ndef generate_records(\n    split: str,\n    examples_per_memory: int,\n    seed: int,\n) -> List[Dict[str, Any]]:\n    records: List[Dict[str, Any]] = []\n    split_offset = {"train": 0, "validation": 1_000_000, "test": 2_000_000}[split]\n    for memory_index, memory_id in enumerate(MEMORY_IDS):\n        rng = np.random.default_rng(seed + split_offset + memory_index * 100_000)\n        labels = np.asarray(\n            [0, 1] * ((examples_per_memory + 1) // 2),\n            dtype=np.int64,\n        )[:examples_per_memory]\n        rng.shuffle(labels)\n        for example_index, label in enumerate(labels):\n            records.append(\n                {\n                    "split": split,\n                    "memory_index": memory_index,\n                    "memory_id": memory_id,\n                    "example_index": example_index,\n                    "prompt": make_example(memory_index, int(label), rng),\n                    "label": int(label),\n                }\n            )\n    random.Random(seed + split_offset).shuffle(records)\n    return records\n\n\ndef save_records(records: List[Dict[str, Any]], path: Path) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", encoding="utf-8") as handle:\n        for record in records:\n            handle.write(json.dumps(record) + "\\n")\n\n\ndef choose_label_tokens(tokenizer: Any) -> Tuple[Tuple[str, str], Tuple[int, int]]:\n    candidates = (\n        (" no", " yes"),\n        (" false", " true"),\n        (" B", " A"),\n        (" 0", " 1"),\n    )\n    for texts in candidates:\n        token_ids = [\n            tokenizer.encode(text, add_special_tokens=False)\n            for text in texts\n        ]\n        if all(len(ids) == 1 for ids in token_ids) and token_ids[0][0] != token_ids[1][0]:\n            return texts, (int(token_ids[0][0]), int(token_ids[1][0]))\n    raise RuntimeError("Could not identify a pair of distinct single-token labels.")\n\n\nclass CausalLabelDataset(Dataset):\n    def __init__(\n        self,\n        records: Sequence[Dict[str, Any]],\n        tokenizer: Any,\n        label_texts: Tuple[str, str],\n        max_length: int,\n    ):\n        self.items: List[Dict[str, torch.Tensor]] = []\n        eos = tokenizer.eos_token_id\n        for record in records:\n            prompt_ids = tokenizer.encode(\n                record["prompt"],\n                add_special_tokens=False,\n                truncation=True,\n                max_length=max_length - 2,\n            )\n            answer_ids = tokenizer.encode(\n                label_texts[int(record["label"])],\n                add_special_tokens=False,\n            )\n            if len(answer_ids) != 1:\n                raise RuntimeError("Label text ceased to be one token.")\n            input_ids = prompt_ids + answer_ids\n            if eos is not None:\n                input_ids += [eos]\n            labels = [-100] * len(prompt_ids) + answer_ids\n            if eos is not None:\n                labels += [-100]\n            self.items.append(\n                {\n                    "input_ids": torch.tensor(input_ids, dtype=torch.long),\n                    "attention_mask": torch.ones(len(input_ids), dtype=torch.long),\n                    "labels": torch.tensor(labels, dtype=torch.long),\n                }\n            )\n\n    def __len__(self) -> int:\n        return len(self.items)\n\n    def __getitem__(self, index: int) -> Dict[str, torch.Tensor]:\n        return self.items[index]\n\n\nclass CausalLabelCollator:\n    def __init__(self, pad_token_id: int):\n        self.pad_token_id = int(pad_token_id)\n\n    def __call__(self, items: Sequence[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:\n        maximum = max(len(item["input_ids"]) for item in items)\n        input_ids = []\n        attention_masks = []\n        labels = []\n        for item in items:\n            padding = maximum - len(item["input_ids"])\n            input_ids.append(\n                F.pad(item["input_ids"], (0, padding), value=self.pad_token_id)\n            )\n            attention_masks.append(\n                F.pad(item["attention_mask"], (0, padding), value=0)\n            )\n            labels.append(F.pad(item["labels"], (0, padding), value=-100))\n        return {\n            "input_ids": torch.stack(input_ids),\n            "attention_mask": torch.stack(attention_masks),\n            "labels": torch.stack(labels),\n        }\n\n\nclass BinaryPromptDataset(Dataset):\n    """Prompt-only examples for restricted two-token next-token training."""\n\n    def __init__(self, records: Sequence[Dict[str, Any]], tokenizer: Any, max_length: int):\n        self.items: List[Dict[str, torch.Tensor]] = []\n        for record in records:\n            encoding = tokenizer(\n                record["prompt"],\n                add_special_tokens=False,\n                truncation=True,\n                max_length=max_length,\n                return_tensors=None,\n            )\n            self.items.append({\n                "input_ids": torch.tensor(encoding["input_ids"], dtype=torch.long),\n                "attention_mask": torch.tensor(encoding["attention_mask"], dtype=torch.long),\n                "binary_label": torch.tensor(int(record["label"]), dtype=torch.long),\n            })\n\n    def __len__(self) -> int:\n        return len(self.items)\n\n    def __getitem__(self, index: int) -> Dict[str, torch.Tensor]:\n        return self.items[index]\n\n\nclass BinaryPromptCollator:\n    def __init__(self, pad_token_id: int):\n        self.pad_token_id = int(pad_token_id)\n\n    def __call__(self, items: Sequence[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:\n        maximum = max(len(item["input_ids"]) for item in items)\n        return {\n            "input_ids": torch.stack([\n                F.pad(item["input_ids"], (0, maximum - len(item["input_ids"])), value=self.pad_token_id)\n                for item in items\n            ]),\n            "attention_mask": torch.stack([\n                F.pad(item["attention_mask"], (0, maximum - len(item["attention_mask"])), value=0)\n                for item in items\n            ]),\n            "binary_label": torch.stack([item["binary_label"] for item in items]),\n        }\n\n\ndef restricted_binary_logits(\n    output_logits: torch.Tensor,\n    attention_mask: torch.Tensor,\n    label_token_ids: Tuple[int, int],\n) -> torch.Tensor:\n    """Return [no, yes] logits at each prompt\'s final unmasked token."""\n    last_indices = attention_mask.sum(dim=1) - 1\n    return output_logits[\n        torch.arange(len(output_logits), device=output_logits.device),\n        last_indices,\n    ][:, list(label_token_ids)]\n\n\n@torch.no_grad()\ndef evaluate_binary_adapter(\n    model: Any,\n    records: Sequence[Dict[str, Any]],\n    tokenizer: Any,\n    label_token_ids: Tuple[int, int],\n    config: Config,\n) -> Dict[str, float]:\n    dataset = BinaryPromptDataset(records, tokenizer, config.max_length)\n    loader = DataLoader(\n        dataset,\n        batch_size=config.inference_batch_size,\n        shuffle=False,\n        collate_fn=BinaryPromptCollator(tokenizer.pad_token_id),\n        pin_memory=True,\n    )\n    model.eval()\n    losses: List[float] = []\n    predictions: List[np.ndarray] = []\n    labels: List[np.ndarray] = []\n    margins: List[np.ndarray] = []\n    for batch in loader:\n        batch = {key: value.to("cuda", non_blocking=True) for key, value in batch.items()}\n        output = model(\n            input_ids=batch["input_ids"],\n            attention_mask=batch["attention_mask"],\n            use_cache=False,\n            return_dict=True,\n        )\n        binary_logits = restricted_binary_logits(\n            output.logits,\n            batch["attention_mask"],\n            label_token_ids,\n        ).float()\n        loss = F.cross_entropy(binary_logits, batch["binary_label"])\n        losses.append(float(loss.item()) * len(binary_logits))\n        predictions.append(binary_logits.argmax(dim=1).cpu().numpy())\n        labels.append(batch["binary_label"].cpu().numpy())\n        margins.append(torch.abs(binary_logits[:, 1] - binary_logits[:, 0]).cpu().numpy())\n    prediction_array = np.concatenate(predictions)\n    label_array = np.concatenate(labels)\n    margin_array = np.concatenate(margins)\n    correct_count = int(\n        np.sum(prediction_array == label_array)\n    )\n    example_count = int(len(label_array))\n    return {\n        "loss": float(sum(losses) / max(example_count, 1)),\n        "accuracy": float(correct_count / max(example_count, 1)),\n        "correct": correct_count,\n        "examples": example_count,\n        "mean_absolute_logit_margin": float(margin_array.mean()),\n        "minimum_absolute_logit_margin": float(margin_array.min()),\n    }\n\n\ndef canonical_base_sha256(model: Any) -> str:\n    """Hash every non-LoRA parameter by canonical name and raw bytes."""\n    hasher = hashlib.sha256()\n    parameters = [\n        (name, parameter)\n        for name, parameter in model.named_parameters()\n        if "lora_" not in name and "modules_to_save" not in name\n    ]\n    for name, parameter in sorted(parameters, key=lambda item: item[0]):\n        hasher.update(name.encode("utf-8"))\n        tensor = parameter.detach().to("cpu").contiguous()\n        hasher.update(str(tensor.dtype).encode("utf-8"))\n        hasher.update(np.asarray(tensor.shape, dtype=np.int64).tobytes())\n        hasher.update(tensor.view(torch.uint8).numpy().tobytes())\n        del tensor\n    return hasher.hexdigest()\n\n\ndef directory_sha256(directory: Path) -> str:\n    hasher = hashlib.sha256()\n    for path in sorted(item for item in directory.rglob("*") if item.is_file()):\n        hasher.update(str(path.relative_to(directory)).encode("utf-8"))\n        with path.open("rb") as handle:\n            while True:\n                chunk = handle.read(1024 * 1024)\n                if not chunk:\n                    break\n                hasher.update(chunk)\n    return hasher.hexdigest()\n\n\ndef assert_only_lora_trainable(model: Any) -> None:\n    invalid = [\n        name for name, parameter in model.named_parameters()\n        if parameter.requires_grad and "lora_" not in name\n    ]\n    if invalid:\n        raise RuntimeError("Non-LoRA parameters became trainable: " + ", ".join(invalid[:10]))\n\n\nclass PromptDataset(Dataset):\n    def __init__(self, records: Sequence[Dict[str, Any]], tokenizer: Any, max_length: int):\n        self.records = list(records)\n        self.encodings = [\n            tokenizer(\n                record["prompt"],\n                add_special_tokens=False,\n                truncation=True,\n                max_length=max_length,\n                return_tensors=None,\n            )\n            for record in self.records\n        ]\n\n    def __len__(self) -> int:\n        return len(self.records)\n\n    def __getitem__(self, index: int) -> Dict[str, Any]:\n        encoding = self.encodings[index]\n        return {\n            "input_ids": torch.tensor(encoding["input_ids"], dtype=torch.long),\n            "attention_mask": torch.tensor(encoding["attention_mask"], dtype=torch.long),\n            "record_index": index,\n        }\n\n\nclass PromptCollator:\n    def __init__(self, pad_token_id: int):\n        self.pad_token_id = int(pad_token_id)\n\n    def __call__(self, items: Sequence[Dict[str, Any]]) -> Dict[str, torch.Tensor]:\n        maximum = max(len(item["input_ids"]) for item in items)\n        return {\n            "input_ids": torch.stack(\n                [\n                    F.pad(item["input_ids"], (0, maximum - len(item["input_ids"])), value=self.pad_token_id)\n                    for item in items\n                ]\n            ),\n            "attention_mask": torch.stack(\n                [\n                    F.pad(item["attention_mask"], (0, maximum - len(item["attention_mask"])), value=0)\n                    for item in items\n                ]\n            ),\n            "record_index": torch.tensor([item["record_index"] for item in items]),\n        }\n\n\ndef load_base_model(config: Config, dtype: torch.dtype) -> Any:\n    model = AutoModelForCausalLM.from_pretrained(\n        config.model_id,\n        torch_dtype=dtype,\n        low_cpu_mem_usage=True,\n    )\n    model.to("cuda")\n    model.config.use_cache = False\n    return model\n\n\ndef assert_cuda_context_healthy() -> None:\n    """Fail early with a useful message when a prior CUDA assert poisoned Colab."""\n    try:\n        probe = torch.tensor([1.0, 2.0], device="cuda")\n        result = (probe * 2.0).sum()\n        torch.cuda.synchronize()\n        if float(result.cpu().item()) != 6.0:\n            raise RuntimeError("Unexpected CUDA health-check result.")\n    except Exception as error:\n        raise RuntimeError(\n            "The CUDA context is unhealthy. A device-side assert permanently "\n            "poisons the current Colab process. Choose Runtime > Restart session, "\n            "then run the notebook again from the first cell."\n        ) from error\n\n\ndef _safe_parameter_sentinel(\n    parameter: torch.Tensor,\n    values_per_tensor: int,\n) -> torch.Tensor:\n    """\n    Copy a few bounded contiguous slices to CPU.\n\n    This deliberately avoids CUDA advanced indexing. The original prototype\n    created a GPU `linspace` index tensor and indexed a flattened parameter,\n    which can trigger a device-side bounds assert on some CUDA/PyTorch builds.\n    """\n    flattened = parameter.detach().reshape(-1)\n    available = int(flattened.numel())\n    requested = min(int(values_per_tensor), available)\n    if requested <= 0:\n        return torch.empty(0, dtype=parameter.dtype)\n\n    segment_count = min(8, requested)\n    segment_width = max(1, math.ceil(requested / segment_count))\n    maximum_start = max(available - segment_width, 0)\n    starts = np.linspace(\n        0,\n        maximum_start,\n        num=segment_count,\n        dtype=np.int64,\n    )\n\n    pieces: List[torch.Tensor] = []\n    for start_value in starts:\n        start = int(start_value)\n        length = min(segment_width, available - start)\n        if length <= 0:\n            continue\n        pieces.append(\n            flattened\n            .narrow(0, start, int(length))\n            .to(device="cpu", non_blocking=False)\n            .clone()\n        )\n\n    if not pieces:\n        return torch.empty(0, dtype=parameter.dtype)\n    return torch.cat(pieces, dim=0)[:requested].contiguous()\n\n\ndef non_lora_sentinels(\n    model: Any,\n    sample_count: int = 6,\n    values_per_tensor: int = 64,\n) -> Dict[str, torch.Tensor]:\n    assert_cuda_context_healthy()\n    candidates = [\n        (name, parameter)\n        for name, parameter in model.named_parameters()\n        if "lora_" not in name\n        and parameter.numel() >= values_per_tensor\n    ]\n    if not candidates:\n        raise RuntimeError("No eligible frozen parameters were found for sentinels.")\n\n    actual_sample_count = min(int(sample_count), len(candidates))\n    selected_indices = np.linspace(\n        0,\n        len(candidates) - 1,\n        num=actual_sample_count,\n        dtype=np.int64,\n    )\n\n    sentinels: Dict[str, torch.Tensor] = {}\n    for selected_index in selected_indices:\n        name, parameter = candidates[int(selected_index)]\n        sentinels[name] = _safe_parameter_sentinel(\n            parameter,\n            values_per_tensor,\n        )\n    return sentinels\n\n\ndef sentinel_retention(\n    reference: Dict[str, torch.Tensor],\n    model: Any,\n) -> float:\n    assert_cuda_context_healthy()\n    current = dict(model.named_parameters())\n    comparisons: List[bool] = []\n    for name, reference_value in reference.items():\n        if name not in current:\n            comparisons.append(False)\n            continue\n        current_value = _safe_parameter_sentinel(\n            current[name],\n            len(reference_value),\n        )\n        comparisons.append(\n            current_value.shape == reference_value.shape\n            and torch.equal(current_value, reference_value)\n        )\n    return float(bool(comparisons) and all(comparisons))\n\n\ndef cosine_warmup(\n    step: int,\n    total_steps: int,\n    warmup_steps: int,\n    minimum_fraction: float,\n) -> float:\n    if step < warmup_steps:\n        return float(step + 1) / max(warmup_steps, 1)\n\n    decay_steps = max(total_steps - warmup_steps, 1)\n    progress = min(\n        max((step - warmup_steps) / decay_steps, 0.0),\n        1.0,\n    )\n    cosine = 0.5 * (\n        1.0 + math.cos(math.pi * progress)\n    )\n    return float(\n        minimum_fraction\n        + (1.0 - minimum_fraction) * cosine\n    )\n\n\ndef train_adapter(\n    base_model: Any,\n    memory_id: str,\n    records: Sequence[Dict[str, Any]],\n    validation_records: Sequence[Dict[str, Any]],\n    tokenizer: Any,\n    label_token_ids: Tuple[int, int],\n    config: Config,\n    adapter_dir: Path,\n    seed: int,\n) -> Tuple[Any, Dict[str, Any]]:\n    adapter_config_path = adapter_dir / "adapter_config.json"\n    adapter_weight_candidates = (\n        adapter_dir / "adapter_model.safetensors",\n        adapter_dir / "adapter_model.bin",\n    )\n    metadata_path = adapter_dir / "dendritron_training.json"\n    if adapter_config_path.exists() and any(path.exists() for path in adapter_weight_candidates) and metadata_path.exists():\n        existing_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))\n        if (\n            existing_metadata.get("objective_version") == OBJECTIVE_VERSION\n            and bool(existing_metadata.get("registered", False))\n            and int(\n                existing_metadata.get(\n                    "best_validation_correct",\n                    -1,\n                )\n            )\n            >= int(\n                existing_metadata.get(\n                    "registration_required_correct",\n                    10**9,\n                )\n            )\n        ):\n            existing_metadata["status"] = "resumed_registered_adapter"\n            existing_metadata["adapter_bytes"] = sum(\n                path.stat().st_size for path in adapter_dir.rglob("*") if path.is_file()\n            )\n            return base_model, existing_metadata\n    if adapter_dir.exists():\n        shutil.rmtree(adapter_dir)\n\n    seed_everything(seed)\n    lora_config = LoraConfig(\n        task_type=TaskType.CAUSAL_LM,\n        r=config.lora_rank,\n        lora_alpha=config.lora_alpha,\n        lora_dropout=config.lora_dropout,\n        target_modules=list(config.lora_targets),\n        bias="none",\n        inference_mode=False,\n    )\n    model = get_peft_model(base_model, lora_config)\n    model.train()\n    model.config.use_cache = False\n    if hasattr(model, "gradient_checkpointing_enable"):\n        model.gradient_checkpointing_enable()\n    if hasattr(model, "enable_input_require_grads"):\n        model.enable_input_require_grads()\n    assert_only_lora_trainable(model)\n    reference_sentinels = non_lora_sentinels(model)\n\n    dataset = BinaryPromptDataset(records, tokenizer, config.max_length)\n    loader = DataLoader(\n        dataset,\n        batch_size=config.batch_size,\n        shuffle=True,\n        collate_fn=BinaryPromptCollator(tokenizer.pad_token_id),\n        pin_memory=True,\n    )\n    trainable = [parameter for parameter in model.parameters() if parameter.requires_grad]\n    optimizer = torch.optim.AdamW(trainable, lr=config.learning_rate, weight_decay=config.weight_decay)\n    total_updates = math.ceil(len(loader) / config.gradient_accumulation) * config.train_epochs\n    warmup_steps = int(total_updates * config.warmup_fraction)\n    scaler_enabled = select_dtype() == torch.float16\n    try:\n        scaler = torch.amp.GradScaler("cuda", enabled=scaler_enabled)\n    except (TypeError, AttributeError):\n        scaler = torch.cuda.amp.GradScaler(enabled=scaler_enabled)\n\n    history: List[Dict[str, float]] = []\n    global_update = 0\n    optimizer.zero_grad(set_to_none=True)\n    started = time.perf_counter()\n    best_validation_accuracy = -1.0\n    best_validation_loss = float("inf")\n    best_validation_correct = -1\n    validation_example_count = len(validation_records)\n    registration_required_correct = int(\n        math.ceil(\n            config.registration_min_accuracy\n            * validation_example_count\n            - 1e-12\n        )\n    )\n    best_epoch = 0\n    epochs_without_improvement = 0\n\n    for epoch in range(1, config.train_epochs + 1):\n        model.train()\n        running_loss = 0.0\n        correct = 0\n        examples = 0\n        for batch_index, batch in enumerate(loader):\n            batch = {key: value.to("cuda", non_blocking=True) for key, value in batch.items()}\n            with torch.autocast(device_type="cuda", dtype=select_dtype(), enabled=True):\n                output = model(\n                    input_ids=batch["input_ids"],\n                    attention_mask=batch["attention_mask"],\n                    use_cache=False,\n                    return_dict=True,\n                )\n                binary_logits = restricted_binary_logits(\n                    output.logits,\n                    batch["attention_mask"],\n                    label_token_ids,\n                ).float()\n                full_loss = F.cross_entropy(binary_logits, batch["binary_label"])\n                loss = full_loss / config.gradient_accumulation\n            scaler.scale(loss).backward()\n            running_loss += float(full_loss.item()) * len(binary_logits)\n            correct += int((binary_logits.argmax(dim=1) == batch["binary_label"]).sum().item())\n            examples += len(binary_logits)\n            should_update = (\n                (batch_index + 1) % config.gradient_accumulation == 0\n                or batch_index + 1 == len(loader)\n            )\n            if should_update:\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(trainable, config.max_grad_norm)\n                scaler.step(optimizer)\n                scaler.update()\n                optimizer.zero_grad(set_to_none=True)\n                learning_rate_scale = cosine_warmup(\n                    global_update,\n                    total_updates,\n                    warmup_steps,\n                    config.minimum_learning_rate_fraction,\n                )\n                for group in optimizer.param_groups:\n                    group["lr"] = (\n                        config.learning_rate\n                        * learning_rate_scale\n                    )\n                global_update += 1\n\n        validation_metrics = evaluate_binary_adapter(\n            model,\n            validation_records,\n            tokenizer,\n            label_token_ids,\n            config,\n        )\n        epoch_row = {\n            "epoch": epoch,\n            "training_loss": running_loss / max(examples, 1),\n            "training_accuracy": correct / max(examples, 1),\n            "validation_loss": validation_metrics["loss"],\n            "validation_accuracy": validation_metrics["accuracy"],\n            "validation_correct": validation_metrics["correct"],\n            "validation_examples": validation_metrics["examples"],\n            "registration_required_correct": registration_required_correct,\n            "validation_mean_margin": validation_metrics["mean_absolute_logit_margin"],\n            "learning_rate": float(optimizer.param_groups[0]["lr"]),\n        }\n        history.append(epoch_row)\n        print(\n            f"{memory_id} epoch {epoch}: "\n            f"train_acc={epoch_row[\'training_accuracy\']:.4f} "\n            f"val_acc={epoch_row[\'validation_accuracy\']:.4f} "\n            f"({epoch_row[\'validation_correct\']}/"\n            f"{epoch_row[\'validation_examples\']}; "\n            f"need {registration_required_correct}) "\n            f"val_loss={epoch_row[\'validation_loss\']:.4f} "\n            f"margin={epoch_row[\'validation_mean_margin\']:.4f}"\n        )\n        improved = (\n            validation_metrics["accuracy"] > best_validation_accuracy + 1e-12\n            or (\n                abs(validation_metrics["accuracy"] - best_validation_accuracy) <= 1e-12\n                and validation_metrics["loss"] < best_validation_loss\n            )\n        )\n        if improved:\n            best_validation_accuracy = validation_metrics["accuracy"]\n            best_validation_loss = validation_metrics["loss"]\n            best_validation_correct = validation_metrics["correct"]\n            best_epoch = epoch\n            epochs_without_improvement = 0\n            adapter_dir.mkdir(parents=True, exist_ok=True)\n            model.save_pretrained(adapter_dir, safe_serialization=True)\n        else:\n            epochs_without_improvement += 1\n        if (\n            epoch >= config.minimum_train_epochs\n            and best_validation_correct >= registration_required_correct\n            and epochs_without_improvement >= config.early_stopping_patience\n        ):\n            print(\n                f"{memory_id}: early stopping after epoch {epoch}; "\n                f"best epoch={best_epoch}, best val={best_validation_accuracy:.4f}"\n            )\n            break\n\n    retention = sentinel_retention(reference_sentinels, model)\n    if retention != 1.0:\n        raise RuntimeError(f"Frozen backbone changed while training {memory_id}.")\n    registered = (\n        best_validation_correct\n        >= registration_required_correct\n    )\n    parameter_counts = {\n        "trainable_parameters": int(sum(parameter.numel() for parameter in trainable)),\n        "total_wrapped_parameters": int(sum(parameter.numel() for parameter in model.parameters())),\n    }\n    metadata = {\n        "memory_id": memory_id,\n        "status": "trained_and_registered" if registered else "rejected",\n        "objective_version": OBJECTIVE_VERSION,\n        "registered": registered,\n        "registration_threshold": config.registration_min_accuracy,\n        "best_epoch": best_epoch,\n        "best_validation_accuracy": best_validation_accuracy,\n        "best_validation_correct": best_validation_correct,\n        "validation_examples": validation_example_count,\n        "registration_required_correct": registration_required_correct,\n        "best_validation_loss": best_validation_loss,\n        "history": history,\n        "formation_seconds": time.perf_counter() - started,\n        "backbone_sentinel_retention": retention,\n        **parameter_counts,\n        "adapter_bytes": sum(path.stat().st_size for path in adapter_dir.rglob("*") if path.is_file()) if adapter_dir.exists() else 0,\n    }\n    adapter_dir.mkdir(parents=True, exist_ok=True)\n    metadata_path = adapter_dir / "dendritron_training.json"\n    metadata_path.write_text(\n        json.dumps(metadata, indent=2),\n        encoding="utf-8",\n    )\n    pd.DataFrame(history).to_csv(\n        adapter_dir / "learning_curve.csv",\n        index=False,\n    )\n\n    base_model = model.unload()\n    base_model.to("cuda")\n    base_model.eval()\n    gc.collect()\n    torch.cuda.empty_cache()\n    if not registered:\n        print(\n            f"{memory_id}: QUARANTINED — best validation "\n            f"{best_validation_correct}/"\n            f"{validation_example_count}; "\n            f"registration requires "\n            f"{registration_required_correct}."\n        )\n    return base_model, metadata\n\n\n@torch.no_grad()\ndef extract_pooled_hidden(\n    model: Any,\n    records: Sequence[Dict[str, Any]],\n    tokenizer: Any,\n    config: Config,\n    tap_index: int,\n) -> np.ndarray:\n    model.eval()\n    dataset = PromptDataset(records, tokenizer, config.max_length)\n    loader = DataLoader(\n        dataset,\n        batch_size=config.coordinate_batch_size,\n        shuffle=False,\n        collate_fn=PromptCollator(tokenizer.pad_token_id),\n        pin_memory=True,\n    )\n    output_values: Optional[np.ndarray] = None\n    for batch in loader:\n        input_ids = batch["input_ids"].to("cuda", non_blocking=True)\n        attention_mask = batch["attention_mask"].to("cuda", non_blocking=True)\n        outputs = model(\n            input_ids=input_ids,\n            attention_mask=attention_mask,\n            output_hidden_states=True,\n            use_cache=False,\n            return_dict=True,\n        )\n        hidden = outputs.hidden_states[tap_index].float()\n        mask = attention_mask.unsqueeze(-1).float()\n        mean_pool = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1.0)\n        last_indices = attention_mask.sum(dim=1) - 1\n        last_pool = hidden[\n            torch.arange(len(hidden), device=hidden.device),\n            last_indices,\n        ]\n        pooled = torch.cat([mean_pool, last_pool], dim=1).cpu().numpy().astype(np.float32)\n        record_indices = batch["record_index"].numpy()\n        if output_values is None:\n            output_values = np.empty((len(records), pooled.shape[1]), dtype=np.float32)\n        output_values[record_indices] = pooled\n    assert output_values is not None\n    return output_values\n\n\nclass PPCA:\n    def __init__(\n        self,\n        mean: np.ndarray,\n        basis: np.ndarray,\n        retained_variance: np.ndarray,\n        residual_variance: float,\n    ):\n        self.mean = np.asarray(mean, dtype=np.float64)\n        self.basis = np.asarray(basis, dtype=np.float64)\n        self.retained_variance = np.asarray(retained_variance, dtype=np.float64)\n        self.residual_variance = float(residual_variance)\n        self.dimension = int(len(self.mean))\n        self.rank = int(self.basis.shape[1])\n        self.log_determinant = float(\n            np.log(self.retained_variance).sum()\n            + (self.dimension - self.rank) * math.log(self.residual_variance)\n        )\n\n    @classmethod\n    def fit(cls, values: np.ndarray, rank: int, floor: float) -> "PPCA":\n        values = values.astype(np.float64)\n        mean = values.mean(axis=0)\n        centered = values - mean\n        covariance = centered.T @ centered / max(len(centered), 1)\n        eigenvalues, eigenvectors = np.linalg.eigh(covariance)\n        order = np.argsort(eigenvalues)[::-1]\n        eigenvalues = eigenvalues[order]\n        eigenvectors = eigenvectors[:, order]\n        retained_rank = min(rank, covariance.shape[0] - 1)\n        residual_variance = max(float(eigenvalues[retained_rank:].mean()), floor)\n        retained_variance = np.maximum(eigenvalues[:retained_rank], residual_variance)\n        return cls(\n            mean,\n            eigenvectors[:, :retained_rank],\n            retained_variance,\n            residual_variance,\n        )\n\n    def log_likelihood(self, values: np.ndarray) -> np.ndarray:\n        difference = values.astype(np.float64) - self.mean\n        total_energy = np.sum(difference * difference, axis=1)\n        projection = difference @ self.basis\n        projected_energy = projection * projection\n        retained_energy = projected_energy.sum(axis=1)\n        mahalanobis = (\n            np.sum(projected_energy / self.retained_variance, axis=1)\n            + (total_energy - retained_energy) / self.residual_variance\n        )\n        return -0.5 * (\n            mahalanobis\n            + self.log_determinant\n            + self.dimension * math.log(2.0 * math.pi)\n        )\n\n\nclass MemoryVerifier:\n    def __init__(self, label_models: Tuple[PPCA, PPCA]):\n        self.label_models = label_models\n\n    @classmethod\n    def fit(\n        cls,\n        coordinates: np.ndarray,\n        labels: np.ndarray,\n        rank: int,\n        floor: float,\n    ) -> "MemoryVerifier":\n        return cls(\n            (\n                PPCA.fit(coordinates[labels == 0], rank, floor),\n                PPCA.fit(coordinates[labels == 1], rank, floor),\n            )\n        )\n\n    def score(self, coordinates: np.ndarray) -> np.ndarray:\n        scores = np.stack(\n            [model.log_likelihood(coordinates) for model in self.label_models],\n            axis=1,\n        )\n        maximum = scores.max(axis=1, keepdims=True)\n        return maximum[:, 0] + np.log(np.exp(scores - maximum).sum(axis=1)) - math.log(2.0)\n\n\ndef fit_shared_temperature(scores: np.ndarray, domains: np.ndarray, config: Config) -> float:\n    score_tensor = torch.from_numpy(scores.astype(np.float32))\n    domain_tensor = torch.from_numpy(domains.astype(np.int64))\n    log_temperature = torch.tensor(0.0, requires_grad=True)\n    optimizer = torch.optim.Adam([log_temperature], lr=config.temperature_learning_rate)\n    for _ in range(config.temperature_steps):\n        optimizer.zero_grad(set_to_none=True)\n        temperature = F.softplus(log_temperature) + 1e-3\n        loss = F.cross_entropy(score_tensor / temperature, domain_tensor)\n        loss.backward()\n        optimizer.step()\n    return float((F.softplus(log_temperature) + 1e-3).detach().item())\n\n\ndef bounded_address_scores(\n    heads: Sequence[LogisticRegression],\n    coordinates: np.ndarray,\n    scale: float,\n) -> np.ndarray:\n    return np.stack(\n        [\n            np.tanh(head.decision_function(coordinates) / scale)\n            for head in heads\n        ],\n        axis=1,\n    ).astype(np.float32)\n\n\ndef smallest_fixed_k(probability: np.ndarray, domains: np.ndarray, target: float) -> int:\n    ranking = np.argsort(-probability, axis=1)\n    for k in range(1, probability.shape[1] + 1):\n        covered = np.any(ranking[:, :k] == domains[:, None], axis=1)\n        if float(covered.mean()) >= target:\n            return k\n    return probability.shape[1]\n\n\ndef finite_sample_quantile(values: Sequence[float], target: float) -> float:\n    array = np.sort(np.asarray(values, dtype=np.float64))\n    count = len(array)\n    level = min(1.0, math.ceil((count + 1) * target) / count)\n    index = min(max(int(math.ceil(level * count)) - 1, 0), count - 1)\n    return float(array[index])\n\n\ndef fit_policies(\n    scores: np.ndarray,\n    domains: np.ndarray,\n    config: Config,\n) -> Dict[str, Any]:\n    temperature = fit_shared_temperature(scores, domains, config)\n    probability = F.softmax(\n        torch.from_numpy(scores) / temperature,\n        dim=1,\n    ).numpy()\n    confidence = probability.max(axis=1)\n    prediction = probability.argmax(axis=1)\n    correct = prediction == domains\n    best = None\n    for threshold in np.unique(confidence):\n        mask = confidence >= threshold\n        if not np.any(mask):\n            continue\n        accuracy = float(correct[mask].mean())\n        coverage = float(mask.mean())\n        if accuracy >= config.efficient_singleton_accuracy:\n            if best is None or coverage > best[0]:\n                best = (coverage, float(threshold))\n    efficient_threshold = float(confidence.max() + 1e-6) if best is None else best[1]\n    reliable_k = smallest_fixed_k(\n        probability,\n        domains,\n        config.reliable_candidate_coverage,\n    )\n\n    conformity = []\n    for row, true_domain in zip(probability, domains):\n        order = np.argsort(-row)\n        true_rank = int(np.where(order == true_domain)[0][0])\n        cumulative = np.cumsum(row[order])\n        conformity.append(\n            float(\n                cumulative[true_rank]\n                + config.raps_penalty\n                * max(true_rank + 1 - config.raps_regularization_rank, 0)\n            )\n        )\n    critical_quantile = finite_sample_quantile(\n        conformity,\n        config.critical_nominal_coverage,\n    )\n    return {\n        "temperature": temperature,\n        "efficient_threshold": efficient_threshold,\n        "reliable_k": int(reliable_k),\n        "critical_quantile": critical_quantile,\n    }\n\n\ndef candidate_sets(\n    probability: np.ndarray,\n    policy: Dict[str, Any],\n    mode: str,\n    config: Config,\n) -> List[List[int]]:\n    ranking = np.argsort(-probability, axis=1)\n    if mode == "fast":\n        return [[int(row[0])] for row in ranking]\n    if mode == "fixed_top2":\n        k = min(2, probability.shape[1])\n        return [[int(value) for value in row[:k]] for row in ranking]\n    if mode == "efficient":\n        result = []\n        for index, row in enumerate(ranking):\n            k = 1 if probability[index, row[0]] >= policy["efficient_threshold"] else min(2, len(row))\n            result.append([int(value) for value in row[:k]])\n        return result\n    if mode == "reliable":\n        k = min(int(policy["reliable_k"]), probability.shape[1])\n        return [[int(value) for value in row[:k]] for row in ranking]\n    if mode == "critical":\n        result = []\n        for row in probability:\n            order = np.argsort(-row)\n            cumulative = 0.0\n            selected = []\n            for rank, memory_index in enumerate(order):\n                selected.append(int(memory_index))\n                cumulative += float(row[memory_index])\n                adjusted = (\n                    cumulative\n                    + config.raps_penalty\n                    * max(rank + 1 - config.raps_regularization_rank, 0)\n                )\n                if adjusted >= policy["critical_quantile"]:\n                    break\n            result.append(selected)\n        return result\n    raise ValueError(mode)\n\n\ndef verifier_matrix(\n    verifiers: Sequence[MemoryVerifier],\n    coordinates: np.ndarray,\n) -> np.ndarray:\n    return np.stack(\n        [verifier.score(coordinates) for verifier in verifiers],\n        axis=1,\n    )\n\n\ndef bind_candidates(\n    verifier_scores: np.ndarray,\n    candidates: Sequence[Sequence[int]],\n) -> np.ndarray:\n    return np.asarray(\n        [\n            max(\n                candidate,\n                key=lambda memory_index: verifier_scores[example_index, memory_index],\n            )\n            for example_index, candidate in enumerate(candidates)\n        ],\n        dtype=np.int64,\n    )\n\n\ndef tokenize_prompts(\n    records: Sequence[Dict[str, Any]],\n    tokenizer: Any,\n    max_length: int,\n    batch_size: int,\n) -> Iterable[Tuple[torch.Tensor, torch.Tensor, np.ndarray]]:\n    dataset = PromptDataset(records, tokenizer, max_length)\n    loader = DataLoader(\n        dataset,\n        batch_size=batch_size,\n        shuffle=False,\n        collate_fn=PromptCollator(tokenizer.pad_token_id),\n        pin_memory=True,\n    )\n    for batch in loader:\n        yield (\n            batch["input_ids"].to("cuda", non_blocking=True),\n            batch["attention_mask"].to("cuda", non_blocking=True),\n            batch["record_index"].numpy(),\n        )\n\n\n@torch.no_grad()\ndef restricted_label_logits(\n    model: Any,\n    records: Sequence[Dict[str, Any]],\n    tokenizer: Any,\n    label_token_ids: Tuple[int, int],\n    config: Config,\n) -> np.ndarray:\n    logits = np.empty((len(records), 2), dtype=np.float32)\n    model.eval()\n    for input_ids, attention_mask, record_indices in tokenize_prompts(\n        records,\n        tokenizer,\n        config.max_length,\n        config.inference_batch_size,\n    ):\n        outputs = model(\n            input_ids=input_ids,\n            attention_mask=attention_mask,\n            use_cache=False,\n            return_dict=True,\n        )\n        binary_logits = restricted_binary_logits(\n            outputs.logits,\n            attention_mask,\n            label_token_ids,\n        )\n        logits[record_indices] = binary_logits.float().cpu().numpy()\n    return logits\n\n\ndef restricted_label_predictions(\n    model: Any,\n    records: Sequence[Dict[str, Any]],\n    tokenizer: Any,\n    label_token_ids: Tuple[int, int],\n    config: Config,\n) -> np.ndarray:\n    return restricted_label_logits(\n        model,\n        records,\n        tokenizer,\n        label_token_ids,\n        config,\n    ).argmax(axis=1)\n\n\n@torch.no_grad()\ndef activate_adapter_for_inference(model: Any, adapter_name: str) -> None:\n    """Activate one adapter without accidentally making it trainable."""\n    try:\n        model.set_adapter(\n            adapter_name,\n            inference_mode=True,\n        )\n    except TypeError:\n        model.set_adapter(adapter_name)\n        if hasattr(model, "set_requires_grad"):\n            model.set_requires_grad(\n                adapter_name,\n                False,\n            )\n        else:\n            for name, parameter in model.named_parameters():\n                if "lora_" in name:\n                    parameter.requires_grad_(False)\n\n\ndef grouped_adapter_logits(\n    model: Any,\n    records: Sequence[Dict[str, Any]],\n    selected_memories: np.ndarray,\n    tokenizer: Any,\n    label_token_ids: Tuple[int, int],\n    config: Config,\n) -> np.ndarray:\n    logits = np.full((len(records), 2), np.nan, dtype=np.float32)\n    for memory_index, memory_id in enumerate(MEMORY_IDS):\n        indices = np.where(selected_memories == memory_index)[0]\n        if len(indices) == 0:\n            continue\n        activate_adapter_for_inference(model, memory_id)\n        subset = [records[int(index)] for index in indices]\n        subset_logits = restricted_label_logits(\n            model,\n            subset,\n            tokenizer,\n            label_token_ids,\n            config,\n        )\n        logits[indices] = subset_logits\n    if np.isnan(logits).any():\n        raise RuntimeError("At least one example was not assigned to an installed memory.")\n    return logits\n\n\ndef grouped_adapter_predictions(\n    model: Any,\n    records: Sequence[Dict[str, Any]],\n    selected_memories: np.ndarray,\n    tokenizer: Any,\n    label_token_ids: Tuple[int, int],\n    config: Config,\n) -> np.ndarray:\n    return grouped_adapter_logits(\n        model,\n        records,\n        selected_memories,\n        tokenizer,\n        label_token_ids,\n        config,\n    ).argmax(axis=1)\n\n\ndef load_all_adapters(base_model: Any, adapters_dir: Path) -> Any:\n    """Load all independent packs into one frozen inference model."""\n    model: Any = PeftModel.from_pretrained(\n        base_model,\n        adapters_dir / MEMORY_IDS[0],\n        adapter_name=MEMORY_IDS[0],\n        is_trainable=False,\n    )\n    for memory_id in MEMORY_IDS[1:]:\n        model.load_adapter(\n            adapters_dir / memory_id,\n            adapter_name=memory_id,\n            is_trainable=False,\n        )\n    activate_adapter_for_inference(\n        model,\n        MEMORY_IDS[0],\n    )\n    model.eval()\n    return model\n\n\ndef package_memory(\n    output_dir: Path,\n    memory_index: int,\n    address_head: LogisticRegression,\n    verifier: MemoryVerifier,\n    pca: PCA,\n    policy: Dict[str, Any],\n    config: Config,\n) -> Path:\n    memory_id = MEMORY_IDS[memory_index]\n    pack_dir = output_dir / "memory_pack_build" / memory_id\n    if pack_dir.exists():\n        shutil.rmtree(pack_dir)\n    pack_dir.mkdir(parents=True, exist_ok=True)\n    adapter_dir = output_dir / "adapters" / memory_id\n    shutil.copytree(adapter_dir, pack_dir / "adapter")\n\n    portable_state: Dict[str, np.ndarray] = {\n        "address_coef": address_head.coef_.astype(np.float32),\n        "address_intercept": address_head.intercept_.astype(np.float32),\n        "address_classes": address_head.classes_.astype(np.int64),\n    }\n    for label_index, model in enumerate(verifier.label_models):\n        portable_state[f"label_{label_index}_mean"] = model.mean.astype(np.float32)\n        portable_state[f"label_{label_index}_basis"] = model.basis.astype(np.float32)\n        portable_state[f"label_{label_index}_retained_variance"] = (\n            model.retained_variance.astype(np.float32)\n        )\n        portable_state[f"label_{label_index}_residual_variance"] = np.asarray(\n            [model.residual_variance],\n            dtype=np.float32,\n        )\n    np.savez_compressed(pack_dir / "memory_state.npz", **portable_state)\n    manifest = {\n        "format": "dendritron-smollm2-memory-v0.4.2",\n        "memory_id": memory_id,\n        "memory_index": memory_index,\n        "base_model": config.model_id,\n        "lora_rank": config.lora_rank,\n        "verifier_rank": config.verifier_rank,\n        "shared_coordinate_required": True,\n        "global_pca_components": int(pca.n_components_),\n        "policy_snapshot": policy,\n        "objective_version": OBJECTIVE_VERSION,\n        "registration_min_accuracy": config.registration_min_accuracy,\n        "raw_examples_stored": 0,\n    }\n    atomic_json_dump(manifest, pack_dir / "manifest.json")\n    zip_path = output_dir / "memory_packs" / f"{memory_id}.dmemory.zip"\n    zip_path.parent.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:\n        for path in pack_dir.rglob("*"):\n            if path.is_file():\n                archive.write(path, arcname=path.relative_to(pack_dir))\n    return zip_path\n\n\ndef build_audit_rows(\n    records: Sequence[Dict[str, Any]],\n    address_scores: np.ndarray,\n    probability: np.ndarray,\n    candidates: Sequence[Sequence[int]],\n    verifier_scores: np.ndarray,\n    selected: np.ndarray,\n    predictions: np.ndarray,\n    count: int = 20,\n) -> List[Dict[str, Any]]:\n    rows = []\n    for index in range(min(count, len(records))):\n        rows.append(\n            {\n                "example_index": index,\n                "memory_id_true": records[index]["memory_id"],\n                "label_true": int(records[index]["label"]),\n                "prompt": records[index]["prompt"],\n                "address_scores": {\n                    MEMORY_IDS[memory_index]: float(address_scores[index, memory_index])\n                    for memory_index in range(len(MEMORY_IDS))\n                },\n                "address_probabilities": {\n                    MEMORY_IDS[memory_index]: float(probability[index, memory_index])\n                    for memory_index in range(len(MEMORY_IDS))\n                },\n                "candidate_memories": [\n                    MEMORY_IDS[memory_index] for memory_index in candidates[index]\n                ],\n                "verifier_scores": {\n                    MEMORY_IDS[memory_index]: float(verifier_scores[index, memory_index])\n                    for memory_index in candidates[index]\n                },\n                "selected_memory": MEMORY_IDS[int(selected[index])],\n                "prediction": int(predictions[index]),\n            }\n        )\n    return rows\n\n\ndef bootstrap_registered_adapters(\n    bootstrap_from_dir: Optional[str],\n    adapters_dir: Path,\n) -> List[Dict[str, Any]]:\n    """Copy and hash-verify compatible registered packs from a prior run."""\n    events: List[Dict[str, Any]] = []\n    if bootstrap_from_dir is None:\n        return events\n\n    source_adapters = Path(bootstrap_from_dir) / "adapters"\n    if not source_adapters.exists():\n        events.append(\n            {\n                "status": "bootstrap_source_missing",\n                "source": str(source_adapters),\n            }\n        )\n        return events\n\n    for memory_id in MEMORY_IDS:\n        source_memory = source_adapters / memory_id\n        target_memory = adapters_dir / memory_id\n        metadata_path = source_memory / "dendritron_training.json"\n\n        if (\n            not source_memory.exists()\n            or not metadata_path.exists()\n            or target_memory.exists()\n        ):\n            continue\n\n        metadata = json.loads(\n            metadata_path.read_text(encoding="utf-8")\n        )\n        compatible = (\n            metadata.get("objective_version") == OBJECTIVE_VERSION\n            and bool(metadata.get("registered", False))\n            and int(metadata.get("best_validation_correct", -1))\n            >= int(metadata.get("registration_required_correct", 10**9))\n        )\n        if not compatible:\n            events.append(\n                {\n                    "memory_id": memory_id,\n                    "status": "bootstrap_incompatible",\n                    "source": str(source_memory),\n                }\n            )\n            continue\n\n        shutil.copytree(source_memory, target_memory)\n        source_hash = directory_sha256(source_memory)\n        target_hash = directory_sha256(target_memory)\n        if source_hash != target_hash:\n            shutil.rmtree(target_memory, ignore_errors=True)\n            raise RuntimeError(\n                f"Bootstrap hash mismatch for {memory_id}."\n            )\n\n        events.append(\n            {\n                "memory_id": memory_id,\n                "status": "bootstrapped_registered_pack",\n                "source": str(source_memory),\n                "target": str(target_memory),\n                "sha256": target_hash,\n                "best_validation_correct": int(\n                    metadata["best_validation_correct"]\n                ),\n                "validation_examples": int(\n                    metadata["validation_examples"]\n                ),\n            }\n        )\n\n    return events\n\n\ndef main(config: Config) -> Dict[str, Any]:\n    config = config.finalized()\n    output_dir = Path(config.output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    adapters_dir = output_dir / "adapters"\n    adapters_dir.mkdir(parents=True, exist_ok=True)\n    started = time.perf_counter()\n\n    bootstrap_events = bootstrap_registered_adapters(\n        config.bootstrap_from_dir,\n        adapters_dir,\n    )\n    atomic_json_dump(\n        {\n            "bootstrap_from_dir": config.bootstrap_from_dir,\n            "events": bootstrap_events,\n        },\n        output_dir / "bootstrap_report.json",\n    )\n\n    seed_everything(config.seed)\n    dtype = select_dtype()\n    assert_cuda_context_healthy()\n    optional_dependency_status = validate_torchao_environment()\n    runtime_environment = gpu_summary()\n    runtime_environment["optional_dependencies"] = optional_dependency_status\n    atomic_json_dump(\n        {\n            "config": asdict(config),\n            "environment": runtime_environment,\n        },\n        output_dir / "run_config.json",\n    )\n    append_progress(\n        output_dir,\n        "start",\n        environment=runtime_environment,\n        bootstrap_events=bootstrap_events,\n    )\n\n    tokenizer = AutoTokenizer.from_pretrained(config.model_id, use_fast=True)\n    if tokenizer.pad_token_id is None:\n        tokenizer.pad_token = tokenizer.eos_token\n    tokenizer.padding_side = "right"\n    label_texts, label_token_ids = choose_label_tokens(tokenizer)\n    atomic_json_dump(\n        {\n            "label_texts": label_texts,\n            "label_token_ids": label_token_ids,\n        },\n        output_dir / "label_tokens.json",\n    )\n\n    train_records = generate_records(\n        "train",\n        config.train_examples_per_memory,\n        config.seed,\n    )\n    validation_records = generate_records(\n        "validation",\n        config.validation_examples_per_memory,\n        config.seed,\n    )\n    test_records = generate_records(\n        "test",\n        config.test_examples_per_memory,\n        config.seed,\n    )\n    save_records(train_records, output_dir / "data" / "train.jsonl")\n    save_records(validation_records, output_dir / "data" / "validation.jsonl")\n    save_records(test_records, output_dir / "data" / "test.jsonl")\n    append_progress(output_dir, "data_ready")\n\n    base_model = load_base_model(config, dtype)\n    append_progress(output_dir, "hashing_initial_backbone")\n    initial_backbone_hash = canonical_base_sha256(base_model)\n    initial_sentinels = non_lora_sentinels(base_model)\n    formation_rows = []\n\n    for memory_index, memory_id in enumerate(MEMORY_IDS):\n        memory_records = [\n            record for record in train_records\n            if record["memory_index"] == memory_index\n        ]\n        memory_validation_records = [\n            record for record in validation_records\n            if record["memory_index"] == memory_index\n        ]\n        base_model, metadata = train_adapter(\n            base_model,\n            memory_id,\n            memory_records,\n            memory_validation_records,\n            tokenizer,\n            label_token_ids,\n            config,\n            adapters_dir / memory_id,\n            config.seed + memory_index * 10_000,\n        )\n        formation_rows.append(metadata)\n        append_progress(\n            output_dir,\n            (\n                "adapter_ready"\n                if metadata["registered"]\n                else "adapter_quarantined"\n            ),\n            memory_id=memory_id,\n            registered=bool(metadata["registered"]),\n            best_validation_correct=int(\n                metadata["best_validation_correct"]\n            ),\n            validation_examples=int(\n                metadata["validation_examples"]\n            ),\n            registration_required_correct=int(\n                metadata[\n                    "registration_required_correct"\n                ]\n            ),\n            adapter_bytes=metadata["adapter_bytes"],\n        )\n\n    formation_df_early = pd.DataFrame(\n        formation_rows\n    )\n    formation_df_early.to_csv(\n        output_dir\n        / "dendritron_smollm2_formation.csv",\n        index=False,\n    )\n\n    backbone_sentinel_retention = sentinel_retention(initial_sentinels, base_model)\n    append_progress(output_dir, "hashing_final_backbone")\n    final_backbone_hash = canonical_base_sha256(base_model)\n    backbone_hash_retention = float(final_backbone_hash == initial_backbone_hash)\n    if (\n        backbone_sentinel_retention != 1.0\n        or backbone_hash_retention != 1.0\n    ):\n        raise RuntimeError(\n            "Frozen backbone integrity check failed "\n            "after memory formation."\n        )\n\n    quarantined = [\n        row\n        for row in formation_rows\n        if not bool(row["registered"])\n    ]\n    if quarantined:\n        diagnostic_payload = {\n            "runtime_version": RUNTIME_VERSION,\n            "objective_version": OBJECTIVE_VERSION,\n            "registration_threshold": (\n                config.registration_min_accuracy\n            ),\n            "quarantined_memories": [\n                {\n                    "memory_id": row["memory_id"],\n                    "best_validation_correct": int(\n                        row["best_validation_correct"]\n                    ),\n                    "validation_examples": int(\n                        row["validation_examples"]\n                    ),\n                    "registration_required_correct": int(\n                        row[\n                            "registration_required_correct"\n                        ]\n                    ),\n                    "best_validation_accuracy": float(\n                        row["best_validation_accuracy"]\n                    ),\n                    "best_epoch": int(row["best_epoch"]),\n                }\n                for row in quarantined\n            ],\n            "backbone_hash_retention": (\n                backbone_hash_retention\n            ),\n        }\n        atomic_json_dump(\n            diagnostic_payload,\n            output_dir\n            / "registration_diagnostics.json",\n        )\n        diagnostic_zip = (\n            output_dir\n            / "Dendritron_v4_1_"\n            "Registration_Diagnostics.zip"\n        )\n        with zipfile.ZipFile(\n            diagnostic_zip,\n            "w",\n            zipfile.ZIP_DEFLATED,\n        ) as archive:\n            archive.write(\n                output_dir\n                / "dendritron_smollm2_formation.csv",\n                arcname=(\n                    "dendritron_smollm2_"\n                    "formation.csv"\n                ),\n            )\n            archive.write(\n                output_dir\n                / "registration_diagnostics.json",\n                arcname=(\n                    "registration_diagnostics.json"\n                ),\n            )\n            for row in quarantined:\n                memory_dir = (\n                    adapters_dir\n                    / row["memory_id"]\n                )\n                for path in memory_dir.rglob("*"):\n                    if path.is_file():\n                        archive.write(\n                            path,\n                            arcname=(\n                                f"quarantined/"\n                                f"{row[\'memory_id\']}/"\n                                f"{path.relative_to(memory_dir)}"\n                            ),\n                        )\n        failed_text = ", ".join(\n            (\n                f"{row[\'memory_id\']} "\n                f"({int(row[\'best_validation_correct\'])}/"\n                f"{int(row[\'validation_examples\'])}; "\n                f"need "\n                f"{int(row[\'registration_required_correct\'])})"\n            )\n            for row in quarantined\n        )\n        raise RuntimeError(\n            "Registry assembly stopped after "\n            "training every pack. Quarantined: "\n            f"{failed_text}. Diagnostics: "\n            f"{diagnostic_zip}"\n        )\n\n    model_layers = int(\n        getattr(\n            base_model.config,\n            "num_hidden_layers",\n        )\n    )\n    tap_index = max(1, min(model_layers, int(round(model_layers * config.tap_fraction))))\n    coordinates_path = output_dir / "coordinates.npz"\n\n    if coordinates_path.exists():\n        coordinate_payload = np.load(coordinates_path)\n        train_pooled = coordinate_payload["train_pooled"]\n        validation_pooled = coordinate_payload["validation_pooled"]\n        test_pooled = coordinate_payload["test_pooled"]\n    else:\n        base_model.eval()\n        train_pooled = extract_pooled_hidden(\n            base_model,\n            train_records,\n            tokenizer,\n            config,\n            tap_index,\n        )\n        validation_pooled = extract_pooled_hidden(\n            base_model,\n            validation_records,\n            tokenizer,\n            config,\n            tap_index,\n        )\n        test_pooled = extract_pooled_hidden(\n            base_model,\n            test_records,\n            tokenizer,\n            config,\n            tap_index,\n        )\n        np.savez_compressed(\n            coordinates_path,\n            train_pooled=train_pooled,\n            validation_pooled=validation_pooled,\n            test_pooled=test_pooled,\n        )\n    append_progress(output_dir, "frozen_hidden_states_ready", tap_index=tap_index)\n\n    pca_path = output_dir / "shared_coordinate_pca.joblib"\n    if pca_path.exists():\n        pca = joblib.load(pca_path)\n    else:\n        pca_components = min(\n            config.pca_dim,\n            train_pooled.shape[0] - 1,\n            train_pooled.shape[1],\n        )\n        pca = PCA(\n            n_components=pca_components,\n            whiten=True,\n            svd_solver="randomized",\n            random_state=config.seed,\n        )\n        pca.fit(train_pooled)\n        joblib.dump(pca, pca_path)\n\n    train_coordinate = pca.transform(train_pooled).astype(np.float32)\n    validation_coordinate = pca.transform(validation_pooled).astype(np.float32)\n    test_coordinate = pca.transform(test_pooled).astype(np.float32)\n\n    train_domains = np.asarray([record["memory_index"] for record in train_records], dtype=np.int64)\n    validation_domains = np.asarray(\n        [record["memory_index"] for record in validation_records],\n        dtype=np.int64,\n    )\n    test_domains = np.asarray([record["memory_index"] for record in test_records], dtype=np.int64)\n    train_labels = np.asarray([record["label"] for record in train_records], dtype=np.int64)\n    validation_labels = np.asarray(\n        [record["label"] for record in validation_records],\n        dtype=np.int64,\n    )\n    test_labels = np.asarray([record["label"] for record in test_records], dtype=np.int64)\n\n    router_path = output_dir / "router_and_verifiers.joblib"\n    if router_path.exists():\n        routing_payload = joblib.load(router_path)\n        address_heads = routing_payload["address_heads"]\n        verifiers = routing_payload["verifiers"]\n        policy = routing_payload["policy"]\n    else:\n        address_heads = []\n        for memory_index in range(len(MEMORY_IDS)):\n            address_head = LogisticRegression(\n                C=2.0,\n                max_iter=2000,\n                solver="lbfgs",\n                class_weight="balanced",\n                random_state=config.seed + memory_index,\n            )\n            address_head.fit(\n                train_coordinate,\n                (train_domains == memory_index).astype(np.int64),\n            )\n            address_heads.append(address_head)\n\n        verifiers = [\n            MemoryVerifier.fit(\n                train_coordinate[train_domains == memory_index],\n                train_labels[train_domains == memory_index],\n                config.verifier_rank,\n                config.covariance_floor,\n            )\n            for memory_index in range(len(MEMORY_IDS))\n        ]\n        validation_address_scores = bounded_address_scores(\n            address_heads,\n            validation_coordinate,\n            config.address_scale,\n        )\n        policy = fit_policies(\n            validation_address_scores,\n            validation_domains,\n            config,\n        )\n        joblib.dump(\n            {\n                "address_heads": address_heads,\n                "verifiers": verifiers,\n                "policy": policy,\n            },\n            router_path,\n        )\n    append_progress(output_dir, "router_and_verifiers_ready", policy=policy)\n\n    for memory_index in range(len(MEMORY_IDS)):\n        package_memory(\n            output_dir,\n            memory_index,\n            address_heads[memory_index],\n            verifiers[memory_index],\n            pca,\n            policy,\n            config,\n        )\n\n    test_address_scores = bounded_address_scores(\n        address_heads,\n        test_coordinate,\n        config.address_scale,\n    )\n    test_probability = F.softmax(\n        torch.from_numpy(test_address_scores) / float(policy["temperature"]),\n        dim=1,\n    ).numpy()\n    test_verifier_scores = verifier_matrix(verifiers, test_coordinate)\n\n    del base_model\n    gc.collect()\n    torch.cuda.empty_cache()\n\n    evaluation_model = load_base_model(config, dtype)\n    evaluation_model.eval()\n    base_predictions = restricted_label_predictions(\n        evaluation_model,\n        test_records,\n        tokenizer,\n        label_token_ids,\n        config,\n    )\n    base_accuracy = float((base_predictions == test_labels).mean())\n\n    # Install packs sequentially and require prior pack outputs to remain exact.\n    old_memory_prediction_retention = 1.0\n    prediction_snapshots: Dict[str, np.ndarray] = {}\n    logit_snapshots: Dict[str, np.ndarray] = {}\n    loaded_memory_ids: List[str] = []\n    for memory_index, memory_id in enumerate(MEMORY_IDS):\n        evaluation_model.load_adapter(\n            adapters_dir / memory_id,\n            adapter_name=memory_id,\n            is_trainable=False,\n        )\n        loaded_memory_ids.append(memory_id)\n        for prior_index, prior_id in enumerate(loaded_memory_ids):\n            indices = np.where(test_domains == prior_index)[0][:80]\n            activate_adapter_for_inference(evaluation_model, prior_id)\n            subset_records = [test_records[int(index)] for index in indices]\n            current_logits = restricted_label_logits(\n                evaluation_model,\n                subset_records,\n                tokenizer,\n                label_token_ids,\n                config,\n            )\n            current = current_logits.argmax(axis=1)\n            if prior_id in prediction_snapshots:\n                old_memory_prediction_retention = min(\n                    old_memory_prediction_retention,\n                    float(np.mean(current == prediction_snapshots[prior_id])),\n                )\n            else:\n                prediction_snapshots[prior_id] = current\n                logit_snapshots[prior_id] = current_logits\n    evaluation_model.eval()\n\n    oracle_predictions = grouped_adapter_predictions(\n        evaluation_model,\n        test_records,\n        test_domains,\n        tokenizer,\n        label_token_ids,\n        config,\n    )\n    oracle_accuracy = float((oracle_predictions == test_labels).mean())\n\n    mode_rows = []\n    mode_payloads: Dict[str, Dict[str, Any]] = {}\n    for mode in ("fast", "efficient", "reliable", "critical"):\n        candidates = candidate_sets(\n            test_probability,\n            policy,\n            mode,\n            config,\n        )\n        selected = bind_candidates(test_verifier_scores, candidates)\n        selected_logits = grouped_adapter_logits(\n            evaluation_model,\n            test_records,\n            selected,\n            tokenizer,\n            label_token_ids,\n            config,\n        )\n        predictions = selected_logits.argmax(axis=1)\n        candidate_coverage = float(\n            np.mean(\n                [\n                    int(test_domains[index]) in candidates[index]\n                    for index in range(len(test_records))\n                ]\n            )\n        )\n        accuracy = float((predictions == test_labels).mean())\n        mode_rows.append(\n            {\n                "mode": mode,\n                "accuracy": accuracy,\n                "oracle_accuracy": oracle_accuracy,\n                "oracle_retention": accuracy / max(oracle_accuracy, 1e-12),\n                "memory_selection_accuracy": float((selected == test_domains).mean()),\n                "candidate_coverage": candidate_coverage,\n                "average_candidates": float(np.mean([len(item) for item in candidates])),\n            }\n        )\n        mode_payloads[mode] = {\n            "candidates": candidates,\n            "selected": selected,\n            "logits": selected_logits,\n            "predictions": predictions,\n        }\n\n    top_two = np.argsort(-test_address_scores, axis=1)[:, :2]\n    address_top2_coverage = float(\n        np.mean(\n            [\n                int(test_domains[index]) in top_two[index]\n                for index in range(len(test_domains))\n            ]\n        )\n    )\n\n    checkpoint_dir = output_dir / "runtime_checkpoint"\n    checkpoint_dir.mkdir(parents=True, exist_ok=True)\n    tokenizer.save_pretrained(checkpoint_dir / "tokenizer")\n    shutil.copy2(pca_path, checkpoint_dir / pca_path.name)\n    shutil.copy2(router_path, checkpoint_dir / router_path.name)\n    atomic_json_dump(\n        {\n            "model_id": config.model_id,\n            "memory_ids": MEMORY_IDS,\n            "tap_index": tap_index,\n            "policy": policy,\n            "label_texts": label_texts,\n            "label_token_ids": label_token_ids,\n        },\n        checkpoint_dir / "runtime_manifest.json",\n    )\n\n    reference_mode = mode_payloads["reliable"]\n    reference_logits = reference_mode["logits"]\n    reference_predictions = reference_mode["predictions"]\n    reference_candidates = reference_mode["candidates"]\n\n    # Reload the base and every independent adapter from their installable directories.\n    del evaluation_model\n    gc.collect()\n    torch.cuda.empty_cache()\n    reloaded_base_model = load_base_model(config, dtype)\n    reloaded_test_pooled = extract_pooled_hidden(\n        reloaded_base_model,\n        test_records,\n        tokenizer,\n        config,\n        tap_index,\n    )\n    reloaded_test_coordinate = pca.transform(\n        reloaded_test_pooled\n    ).astype(np.float32)\n    reloaded_address_scores = bounded_address_scores(\n        address_heads,\n        reloaded_test_coordinate,\n        config.address_scale,\n    )\n    reloaded_probability = F.softmax(\n        torch.from_numpy(reloaded_address_scores)\n        / float(policy["temperature"]),\n        dim=1,\n    ).numpy()\n    reloaded_candidates = candidate_sets(\n        reloaded_probability,\n        policy,\n        "reliable",\n        config,\n    )\n    reloaded_verifier_scores = verifier_matrix(\n        verifiers,\n        reloaded_test_coordinate,\n    )\n    reloaded_selected = bind_candidates(\n        reloaded_verifier_scores,\n        reloaded_candidates,\n    )\n    reloaded_model = load_all_adapters(\n        reloaded_base_model,\n        adapters_dir,\n    )\n    reloaded_model.eval()\n    reloaded_logits = grouped_adapter_logits(\n        reloaded_model,\n        test_records,\n        reloaded_selected,\n        tokenizer,\n        label_token_ids,\n        config,\n    )\n    reloaded_predictions = reloaded_logits.argmax(axis=1)\n    checkpoint_prediction_equivalence = float(\n        np.mean(reloaded_predictions == reference_predictions)\n    )\n    checkpoint_max_logit_delta = float(\n        np.max(np.abs(reloaded_logits - reference_logits))\n    )\n    checkpoint_candidate_equivalence = float(\n        np.mean(\n            [\n                reloaded_candidates[index]\n                == reference_candidates[index]\n                for index in range(\n                    len(reference_candidates)\n                )\n            ]\n        )\n    )\n\n    # Physical adapter deletion and reload.\n    removed_memory = MEMORY_IDS[2]\n    removed_adapter_dir = adapters_dir / removed_memory\n    adapter_hash_before = directory_sha256(removed_adapter_dir)\n    reloaded_model.delete_adapter(removed_memory)\n    uninstall_selection_exclusion = float(\n        removed_memory not in set(reloaded_model.peft_config.keys())\n    )\n    reloaded_model.load_adapter(\n        removed_adapter_dir,\n        adapter_name=removed_memory,\n        is_trainable=False,\n    )\n    adapter_hash_after = directory_sha256(removed_adapter_dir)\n    adapter_hash_equivalence = float(adapter_hash_before == adapter_hash_after)\n    reinstall_indices = np.where(test_domains == 2)[0][:80]\n    reinstall_records = [test_records[int(index)] for index in reinstall_indices]\n    activate_adapter_for_inference(reloaded_model, removed_memory)\n    after_reinstall_logits = restricted_label_logits(\n        reloaded_model,\n        reinstall_records,\n        tokenizer,\n        label_token_ids,\n        config,\n    )\n    after_reinstall = after_reinstall_logits.argmax(axis=1)\n    reinstall_prediction_equivalence = float(\n        np.mean(after_reinstall == prediction_snapshots[removed_memory])\n    )\n    reinstall_max_logit_delta = float(\n        np.max(np.abs(after_reinstall_logits - logit_snapshots[removed_memory]))\n    )\n\n    critical_payload = mode_payloads["critical"]\n    audit_rows = build_audit_rows(\n        test_records,\n        test_address_scores,\n        test_probability,\n        critical_payload["candidates"],\n        test_verifier_scores,\n        critical_payload["selected"],\n        critical_payload["predictions"],\n    )\n    atomic_json_dump({"traces": audit_rows}, output_dir / "audit_trace.json")\n\n    formation_df = pd.DataFrame(formation_rows)\n    mode_df = pd.DataFrame(mode_rows)\n    adapter_parameters = int(formation_df["trainable_parameters"].mean())\n    adapter_bytes = float(formation_df["adapter_bytes"].mean())\n\n    reliable_row = mode_df.set_index("mode").loc["reliable"]\n    critical_row = mode_df.set_index("mode").loc["critical"]\n    minimum_pack_validation_accuracy = float(\n        formation_df["best_validation_accuracy"].min()\n    )\n    results = {\n        "model": "Dendritron Runtime v0.4.2 on SmolLM2-360M",\n        "runtime_version": RUNTIME_VERSION,\n        "objective_version": OBJECTIVE_VERSION,\n        "base_model": config.model_id,\n        "quick_mode": config.quick_mode,\n        "bootstrap_from_dir": config.bootstrap_from_dir,\n        "bootstrapped_memories": int(\n            sum(\n                event.get("status")\n                == "bootstrapped_registered_pack"\n                for event in bootstrap_events\n            )\n        ),\n        "memories": len(MEMORY_IDS),\n        "base_accuracy": base_accuracy,\n        "oracle_accuracy": oracle_accuracy,\n        "reliable_accuracy": float(reliable_row["accuracy"]),\n        "critical_accuracy": float(critical_row["accuracy"]),\n        "critical_oracle_retention": float(critical_row["oracle_retention"]),\n        "critical_candidate_coverage": float(critical_row["candidate_coverage"]),\n        "critical_average_candidates": float(critical_row["average_candidates"]),\n        "minimum_pack_validation_accuracy": minimum_pack_validation_accuracy,\n        "address_top2_coverage": address_top2_coverage,\n        "old_memory_prediction_retention": old_memory_prediction_retention,\n        "backbone_sentinel_retention": backbone_sentinel_retention,\n        "backbone_hash_retention": backbone_hash_retention,\n        "initial_backbone_sha256": initial_backbone_hash,\n        "final_backbone_sha256": final_backbone_hash,\n        "checkpoint_prediction_equivalence": checkpoint_prediction_equivalence,\n        "checkpoint_candidate_equivalence": checkpoint_candidate_equivalence,\n        "checkpoint_max_logit_delta": checkpoint_max_logit_delta,\n        "uninstall_selection_exclusion": uninstall_selection_exclusion,\n        "reinstall_prediction_equivalence": reinstall_prediction_equivalence,\n        "reinstall_max_logit_delta": reinstall_max_logit_delta,\n        "adapter_hash_equivalence": adapter_hash_equivalence,\n        "adapter_parameters_per_memory": adapter_parameters,\n        "adapter_bytes_per_memory": adapter_bytes,\n        "shared_coordinate_dimension": int(pca.n_components_),\n        "tap_index": tap_index,\n        "raw_examples_stored_in_memory_pack": 0,\n        "runtime_seconds": time.perf_counter() - started,\n    }\n    gate_pass = bool(\n        results["minimum_pack_validation_accuracy"] >= GATE_THRESHOLDS["minimum_pack_validation_accuracy"]\n        and results["oracle_accuracy"] >= GATE_THRESHOLDS["oracle_accuracy"]\n        and results["reliable_accuracy"] >= GATE_THRESHOLDS["reliable_accuracy"]\n        and results["critical_accuracy"] >= GATE_THRESHOLDS["critical_accuracy"]\n        and results["critical_oracle_retention"] >= GATE_THRESHOLDS["critical_oracle_retention"]\n        and results["critical_average_candidates"] <= GATE_THRESHOLDS["critical_average_candidates"]\n        and results["address_top2_coverage"] >= GATE_THRESHOLDS["address_top2_coverage"]\n        and results["old_memory_prediction_retention"] >= GATE_THRESHOLDS["old_memory_prediction_retention"]\n        and results["backbone_hash_retention"] >= GATE_THRESHOLDS["backbone_hash_retention"]\n        and results["checkpoint_prediction_equivalence"] >= GATE_THRESHOLDS["checkpoint_prediction_equivalence"]\n        and results["checkpoint_candidate_equivalence"] >= GATE_THRESHOLDS["checkpoint_candidate_equivalence"]\n        and results["checkpoint_max_logit_delta"] <= GATE_THRESHOLDS["checkpoint_max_logit_delta"]\n        and results["uninstall_selection_exclusion"] >= GATE_THRESHOLDS["uninstall_selection_exclusion"]\n        and results["reinstall_prediction_equivalence"] >= GATE_THRESHOLDS["reinstall_prediction_equivalence"]\n        and results["reinstall_max_logit_delta"] <= GATE_THRESHOLDS["reinstall_max_logit_delta"]\n        and results["adapter_hash_equivalence"] >= GATE_THRESHOLDS["adapter_hash_equivalence"]\n    )\n    results["gate_pass"] = gate_pass\n\n    pd.DataFrame([results]).to_csv(output_dir / "dendritron_smollm2_results.csv", index=False)\n    mode_df.to_csv(output_dir / "dendritron_smollm2_modes.csv", index=False)\n    formation_df.to_csv(output_dir / "dendritron_smollm2_formation.csv", index=False)\n    atomic_json_dump(\n        {\n            "config": asdict(config),\n            "thresholds": GATE_THRESHOLDS,\n            "environment": runtime_environment,\n            "policy": policy,\n            "results": results,\n        },\n        output_dir / "dendritron_smollm2_summary.json",\n    )\n    append_progress(output_dir, "complete", gate_pass=gate_pass)\n\n    final_zip = output_dir / "Dendritron_SmolLM2_360M_Results_and_Memory_Packs.zip"\n    with zipfile.ZipFile(final_zip, "w", zipfile.ZIP_DEFLATED) as archive:\n        for path in output_dir.rglob("*"):\n            if not path.is_file() or path == final_zip:\n                continue\n            # Do not duplicate the large combined checkpoint model in the compact handoff zip.\n            if "runtime_checkpoint/model_with_adapters" in str(path):\n                continue\n            archive.write(path, arcname=path.relative_to(output_dir))\n\n    print("\\nFINAL RESULTS")\n    print(pd.DataFrame([results]).to_string(index=False))\n    print("\\nMODE SUMMARY")\n    print(mode_df.to_string(index=False))\n    print(f"\\nArtifacts: {output_dir}")\n    print(f"Compact package: {final_zip}")\n    return {\n        "results": results,\n        "modes": mode_df,\n        "formation": formation_df,\n        "output_dir": str(output_dir),\n        "zip": str(final_zip),\n    }\n\n\ndef parse_args() -> Config:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--output-dir", default="/content/dendritron_smollm2_360m_v4_2")\n    parser.add_argument("--seed", type=int, default=7)\n    parser.add_argument("--quick-mode", action="store_true")\n    arguments = parser.parse_args()\n    return Config(\n        output_dir=arguments.output_dir,\n        seed=arguments.seed,\n        quick_mode=arguments.quick_mode,\n    )\n\n\nif __name__ == "__main__":\n    import traceback\n\n    parsed_config = parse_args()\n    try:\n        main(parsed_config)\n    except Exception as error:\n        failure_dir = Path(parsed_config.output_dir)\n        failure_dir.mkdir(parents=True, exist_ok=True)\n        failure_payload = {\n            "error_type": type(error).__name__,\n            "error": str(error),\n            "traceback": traceback.format_exc(),\n        }\n        atomic_json_dump(\n            failure_payload,\n            failure_dir / "failure.json",\n        )\n        print("\\nDENDRITRON RUN FAILED\\n")\n        traceback.print_exc()\n        print(\n            "\\nFailure details saved to:",\n            failure_dir / "failure.json",\n        )\n        raise\n'
SCRIPT_PATH = Path("/content/dendritron_smollm2_360m_v4_2.py")
SCRIPT_PATH.write_text(BENCHMARK_SOURCE, encoding="utf-8")
compile(BENCHMARK_SOURCE, str(SCRIPT_PATH), "exec")

print("Implementation written to:", SCRIPT_PATH)
print("Source characters:", len(BENCHMARK_SOURCE))


In [ ]:
# Import directly into this notebook kernel.
import importlib.util
import sys

module_name = "dendritron_smollm2_v4_2"
spec = importlib.util.spec_from_file_location(
    module_name,
    SCRIPT_PATH,
)
dendritron = importlib.util.module_from_spec(spec)
sys.modules[module_name] = dendritron
spec.loader.exec_module(dendritron)

config = dendritron.Config(
    output_dir=OUTPUT_DIR,
    seed=7,
    quick_mode=QUICK_MODE,
    bootstrap_from_dir=BOOTSTRAP_DIR,
).finalized()

print(config)


In [ ]:
# Preview the prompts and exact binary semantics before training.
from IPython.display import display
import pandas as pd

preview_records = dendritron.generate_records(
    "train",
    examples_per_memory=6,
    seed=config.seed,
)
preview = pd.DataFrame(preview_records)[
    ["memory_id", "prompt", "label"]
]
preview["semantic_answer"] = preview["label"].map({0: "no", 1: "yes"})
display(preview)


## Repair the registry and run the complete gate

When the v0.4.1 directory is present, four registered packs are copied and
hash-verified, then their training is skipped. Only `endpoint_match` is newly
trained.

When the prior directory is absent, the notebook remains fully self-contained
and trains all five packs.

After registration, Dendritron evaluates autonomous candidate proposal,
generative binding, oracle retention, checkpoint routing and logits, physical
adapter deletion, reinstall, and adapter-file hashes.


In [ ]:
# Execute with complete registration diagnostics.
import traceback
from pathlib import Path
from IPython.display import display
import pandas as pd

try:
    artifacts = dendritron.main(config)
except Exception:
    traceback.print_exc()
    output_path = Path(OUTPUT_DIR)
    progress_path = output_path / "progress.json"
    formation_path = (
        output_path
        / "dendritron_smollm2_formation.csv"
    )
    diagnostics_path = (
        output_path
        / "registration_diagnostics.json"
    )
    diagnostic_zip = (
        output_path
        / "Dendritron_v4_1_"
        "Registration_Diagnostics.zip"
    )

    if progress_path.exists():
        print("\nLast recorded progress:")
        print(
            progress_path.read_text(
                encoding="utf-8"
            )
        )

    if formation_path.exists():
        print("\nAll memory-learning results:")
        display(pd.read_csv(formation_path))

    if diagnostics_path.exists():
        print("\nRegistration diagnostics:")
        print(
            diagnostics_path.read_text(
                encoding="utf-8"
            )
        )

    if diagnostic_zip.exists():
        print(
            "\nDiagnostic package:",
            diagnostic_zip,
        )

    raise


## Read the result in layers

First require content acquisition:

\[
\min_t \operatorname{ValAccuracy}(M_t) \ge 80\%.
\]

Then compare autonomous Dendritron performance with oracle adapter selection.

In [ ]:
# Display content acquisition and autonomous recall separately.
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

results = pd.read_csv(Path(OUTPUT_DIR) / "dendritron_smollm2_results.csv")
modes = pd.read_csv(Path(OUTPUT_DIR) / "dendritron_smollm2_modes.csv")
formation = pd.read_csv(Path(OUTPUT_DIR) / "dendritron_smollm2_formation.csv")

print("System result")
display(results)
print("Functional memory acquisition")
display(formation[[
    "memory_id", "status", "best_epoch", "best_validation_accuracy",
    "best_validation_loss", "formation_seconds", "trainable_parameters",
    "adapter_bytes", "backbone_sentinel_retention",
]])
print("Autonomous reliability / compute")
display(modes)

figure = plt.figure(figsize=(8, 5))
axis = figure.add_subplot(111)
axis.scatter(modes["average_candidates"], modes["accuracy"], s=90)
for _, row in modes.iterrows():
    axis.annotate(row["mode"], (row["average_candidates"], row["accuracy"]), xytext=(5, 5), textcoords="offset points")
axis.axhline(float(results.iloc[0]["oracle_accuracy"]), linestyle="--", label="oracle adapter")
axis.set_xlabel("Average memory packs evaluated")
axis.set_ylabel("End-to-end accuracy")
axis.set_title("Dendritron v0.4 reliability / compute frontier")
axis.legend()
figure.show()


## Inspect a Dendritron decision

A trace records:

- every bounded address score;
- the calibrated address posterior;
- the candidate packs preserved by the reliability policy;
- each candidate's generative likelihood;
- the selected memory;
- the final binary answer.

This is the basis for auditable functional memory.


In [ ]:
# Explore one complete critical-mode audit trace.
import json
from pathlib import Path
from pprint import pprint

audit = json.loads(
    (Path(OUTPUT_DIR) / "audit_trace.json").read_text(encoding="utf-8")
)
pprint(audit["traces"][0], sort_dicts=False)


In [ ]:
# Show the installable memory packs produced by the run.
from pathlib import Path
import zipfile
from IPython.display import display
import pandas as pd

pack_dir = Path(OUTPUT_DIR) / "memory_packs"
pack_rows = []
for pack_path in sorted(pack_dir.glob("*.dmemory.zip")):
    with zipfile.ZipFile(pack_path) as archive:
        names = archive.namelist()
    pack_rows.append(
        {
            "memory_pack": pack_path.name,
            "size_mb": round(pack_path.stat().st_size / 1024**2, 3),
            "contains_adapter": any(name.startswith("adapter/") for name in names),
            "contains_portable_state": "memory_state.npz" in names,
            "contains_manifest": "manifest.json" in names,
        }
    )
display(pd.DataFrame(pack_rows))


In [ ]:
# Download the compact results and all five installable packs.
from pathlib import Path

zip_path = (
    Path(OUTPUT_DIR)
    / "Dendritron_SmolLM2_360M_Results_and_Memory_Packs.zip"
)
print("Package:", zip_path)

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print("Use the Colab Files sidebar to download the package.")
